# Free-Tier QLoRA Fine-Tuning Pipeline (Kaggle / Colab)

This standalone notebook performs **parameter-efficient 4-bit QLoRA fine-tuning** on open-weights language models (e.g., `Qwen/Qwen2.5-7B-Instruct` or `meta-llama/Llama-3.1-8B-Instruct`) optimized for free 16GB GPU runtimes (Kaggle Dual T4 / Colab T4) and streams final checkpoints to Google Cloud Storage.

### Target Architecture:
- **Base Model**: 4-bit NF4 Quantization with double quantization via `bitsandbytes` (~5.5GB VRAM footprint).
- **PEFT / LoRA**: Attention projection target modules `['q_proj', 'v_proj', 'k_proj', 'o_proj']` ($r=16, \alpha=32$).
- **Trainer**: Hugging Face `trl` `SFTTrainer` with `paged_adamw_8bit` and gradient checkpointing.
- **Artifacts**: LoRA adapters saved locally and streamed directly to a 5TB Google Cloud Storage bucket.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q --upgrade transformers peft trl bitsandbytes accelerate datasets google-cloud-storage

In [ ]:
# Step 2: GPU Diagnostics
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM: {vram:.2f} GB")
else:
    print("Running on CPU. In Kaggle, enable GPU in Settings -> Accelerator -> GPU T4 x2. In Colab, Runtime -> Change runtime type -> T4.")

In [ ]:
# Step 3: Secrets & Cloud Credentials
import os

# Set GCS bucket
GCS_BUCKET = os.environ.get("GCS_BUCKET_NAME", "agi-agent-ingestion-data")

# Hugging Face Token (required only if using gated models like Llama-3.1-8B)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    pass

print(f"GCS Target Bucket: {GCS_BUCKET}")

In [ ]:
# Step 4: Dataset Preparation (ChatML Research Pairs)
from datasets import Dataset

sample_data = [
    {
        "instruction": "Analyze the research excerpt, synthesize its key contributions, and architectural implications.",
        "input": "Paper: Coding Agents with an Obstacle-Aware Harness for Safe Robot Manipulation\nSource: arxiv (Authors: Bingxin Xu, et al.)\n\nExcerpt: Coding agents write controllers as programs. SafeHarness attains 71.9% task success and 87.5% collision avoidance.",
        "output": "### Technical Summary\n1. **Core Contribution**: Introduces SafeHarness for constraint-aware robot controller synthesis.\n2. **Architecture**: Decomposes manipulation into route planning and obstacle-aware contact execution.\n3. **Empirical Results**: 87.5% collision avoidance (27% improvement over baseline).",
        "text": "<|im_start|>system\nAnalyze the research excerpt, synthesize its key contributions, and architectural implications.<|im_end|>\n<|im_start|>user\nPaper: Coding Agents with an Obstacle-Aware Harness for Safe Robot Manipulation\nSource: arxiv (Authors: Bingxin Xu, et al.)\n\nExcerpt: Coding agents write controllers as programs. SafeHarness attains 71.9% task success and 87.5% collision avoidance.<|im_end|>\n<|im_start|>assistant\n### Technical Summary\n1. **Core Contribution**: Introduces SafeHarness for constraint-aware robot controller synthesis.\n2. **Architecture**: Decomposes manipulation into route planning and obstacle-aware contact execution.\n3. **Empirical Results**: 87.5% collision avoidance (27% improvement over baseline).<|im_end|>"
    },
    {
        "instruction": "Analyze the research excerpt, synthesize its key contributions, and architectural implications.",
        "input": "Paper: Scaling Law Limits in Cyclic Reasoning Agents\nSource: arxiv (Authors: A. Vaswani, D. Silver)\n\nExcerpt: By incorporating sandboxed Python execution and AST security reflection, task completion rates improve by 38.4% over single-shot prompting.",
        "output": "### Technical Summary\n1. **Core Contribution**: Demonstrates compute scaling via cyclic ReAct loops.\n2. **Architecture**: Sandboxed execution with static AST safety screening and automated retry reflection.\n3. **Empirical Results**: 38.4% improvement in task completion.",
        "text": "<|im_start|>system\nAnalyze the research excerpt, synthesize its key contributions, and architectural implications.<|im_end|>\n<|im_start|>user\nPaper: Scaling Law Limits in Cyclic Reasoning Agents\nSource: arxiv (Authors: A. Vaswani, D. Silver)\n\nExcerpt: By incorporating sandboxed Python execution and AST security reflection, task completion rates improve by 38.4% over single-shot prompting.<|im_end|>\n<|im_start|>assistant\n### Technical Summary\n1. **Core Contribution**: Demonstrates compute scaling via cyclic ReAct loops.\n2. **Architecture**: Sandboxed execution with static AST safety screening and automated retry reflection.\n3. **Empirical Results**: 38.4% improvement in task completion.<|im_end|>"
    }
]

dataset = Dataset.from_list(sample_data)
print(f"Prepared {len(dataset)} training samples.")

In [ ]:
# Step 5: Load 4-Bit Base Model and Setup PEFT / LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # or "meta-llama/Llama-3.1-8B-Instruct"
OUTPUT_DIR = "./lora_checkpoints"

compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(f"Loading {MODEL_ID} in 4-bit NF4...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("Model and LoRA layers ready!")

In [ ]:
# Step 6: Fine-Tune with SFTTrainer (Compatible with trl SFTConfig)
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    max_steps=20,
    logging_steps=5,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    save_strategy="epoch",
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting fine-tuning...")
trainer.train()
print("Training completed!")

# Save trained LoRA adapters
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter artifacts to {OUTPUT_DIR}")

In [ ]:
# Step 7: Stream Checkpoint Artifacts to 5TB GCS Bucket
from google.cloud import storage
from pathlib import Path
from datetime import datetime, timezone

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
remote_prefix = f"models/lora_checkpoints/run_{run_timestamp}"

try:
    client = storage.Client()
    bucket = client.bucket(GCS_BUCKET)
    local_path = Path(OUTPUT_DIR)
    for fp in local_path.rglob("*"):
        if fp.is_file():
            rel = fp.relative_to(local_path).as_posix()
            blob_name = f"{remote_prefix}/{rel}"
            blob = bucket.blob(blob_name)
            blob.upload_from_filename(str(fp))
            print(f"Uploaded: gs://{GCS_BUCKET}/{blob_name}")
    print(f"All checkpoint artifacts synced to gs://{GCS_BUCKET}/{remote_prefix}/")
except Exception as e:
    print(f"GCS Upload Note: {e}")
    print("To enable streaming to GCS, provide your Google Cloud service account JSON or configure ADC.")